# R3maJ — W&B Training on Colab

W&B-enabled training supervisor for R3maJ. Uses Colab Secrets for the W&B API key, parses training metrics from `train.log`, logs them against global training timesteps, and supports automatic checkpoint backup/restart.

**Before running:** add a Colab Secret named `WANDB_API_KEY`. Never hard-code the API key into the notebook.

In [ ]:
# W&B setup
import os, subprocess
from google.colab import userdata

subprocess.run(['pip', 'install', '-q', 'wandb'], check=True)
import wandb

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
if not WANDB_API_KEY:
    raise RuntimeError('WANDB_API_KEY not found. Add it in Colab -> Secrets.')

os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_MODE'] = 'online'

run = wandb.init(
    project='R3maJ',
    name='R3maJ-training',
    resume='allow',
    config={
        'architecture': '512x6',
        'games': 164,
        'device': 'cuda',
        'tick_skip': 8,
        'action_delay': 2,
    },
)
print('wandb:', wandb.__version__)
print('W&B run:', run.url)

In [ ]:
# Training metric parser
import re, threading, time
latest_metrics = {}

def parse_training_line(line):
    patterns = {
        'train/step_reward': r'Average Step Reward\s*: ?\s*([0-9.eE+-]+)',
        'train/policy_entropy': r'Policy Entropy\s*: ?\s*([0-9.eE+-]+)',
        'train/clipped_reward': r'Clipped Reward Portion\s*: ?\s*([0-9.eE+-]+)',
        'train/total_timesteps': r'Total Timesteps\s*: ?\s*([0-9.eE+-]+)',
        'train/iteration': r'Total Iterations\s*: ?\s*([0-9.eE+-]+)',
        'performance/collection_sps': r'Collection Steps/Second\s*: ?\s*([0-9.eE+-]+)',
        'performance/consumption_sps': r'Consumption Steps/Second\s*: ?\s*([0-9.eE+-]+)',
        'performance/collection_time': r'Collection Time\s*: ?\s*([0-9.eE+-]+)',
        'performance/consumption_time': r'Consumption Time\s*: ?\s*([0-9.eE+-]+)',
        'ball/angular_speed': r'Ball/AngularSpeed\s*: ?\s*([0-9.eE+-]+)',
        'ball/height': r'Ball/Height\s*: ?\s*([0-9.eE+-]+)',
        'ball/speed': r'Ball/Speed\s*: ?\s*([0-9.eE+-]+)',
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, line)
        if m:
            latest_metrics[key] = float(m.group(1))

In [ ]:
# W&B training logger
def get_phase(timesteps):
    if timesteps < 5_000_000_000: return 0
    if timesteps < 15_000_000_000: return 1
    if timesteps < 30_000_000_000: return 2
    return 3

def wandb_logger():
    last_step = -1
    while True:
        time.sleep(5)
        if not latest_metrics: continue
        metrics = dict(latest_metrics)
        step = metrics.get('train/total_timesteps')
        if step is None: continue
        step = int(step)
        if step <= last_step: continue
        last_step = step
        metrics['curriculum/phase'] = get_phase(step)
        try:
            wandb.log(metrics, step=step)
        except Exception as e:
            print('[wandb] log error:', e, flush=True)

if '_R3MAJ_WANDB_LOGGER_' not in globals():
    globals()['_R3MAJ_WANDB_LOGGER_'] = True
    threading.Thread(target=wandb_logger, daemon=True).start()
    print('W&B training logger started.')

In [ ]:
# Training supervisor
import os, sys, subprocess, torch, time, threading, shutil, glob

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda': print('GPU detected:', torch.cuda.get_device_name(0), flush=True)
else: print('WARNING: no GPU detected - training on CPU', flush=True)

BUILD = '/content/R3maJ/build'
LOCAL_CKPT = os.path.join(BUILD, 'checkpoints')
DRIVE_CKPT = '/content/drive/MyDrive/R3maJ/checkpoints'
os.makedirs(LOCAL_CKPT, exist_ok=True)

def _ts(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

def _backup_once():
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    local = sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT, '*')) if os.path.isdir(d)], key=_ts)
    drive = set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT, '*')))
    for d in local:
        name = os.path.basename(d)
        if name in drive: continue
        tmp = os.path.join(DRIVE_CKPT, name + '.tmp')
        try:
            print(f'[backup] uploading checkpoint {name} ...', flush=True)
            shutil.copytree(d, tmp)
            shutil.move(tmp, os.path.join(DRIVE_CKPT, name))
            print(f'[backup] {name} uploaded', flush=True)
        except Exception as e:
            print('[backup] err:', e, flush=True)
            shutil.rmtree(tmp, ignore_errors=True)

def _backup_loop():
    while True:
        try:
            if os.path.isdir('/content/drive'): _backup_once()
        except Exception as e: print('[backup] loop err:', e, flush=True)
        time.sleep(60)

if '_R3MAJ_BACKUP_DAEMON_' not in globals():
    globals()['_R3MAJ_BACKUP_DAEMON_'] = True
    threading.Thread(target=_backup_loop, daemon=True).start()

_tailer_lock = {'run': True}
if '_R3MAJ_TAILER_' not in globals():
    globals()['_R3MAJ_TAILER_'] = True
    def _tail_loop():
        path = os.path.join(BUILD, 'train.log')
        while not os.path.exists(path): time.sleep(1)
        with open(path, 'r', errors='ignore') as f:
            f.seek(0, os.SEEK_END)
            while True:
                try:
                    data = f.read()
                    if data:
                        sys.stdout.write(data); sys.stdout.flush()
                        for line in data.splitlines(): parse_training_line(line)
                    else:
                        if not _tailer_lock.get('run', True): return
                        time.sleep(0.5)
                except Exception as e:
                    print('[tailer] error:', e, flush=True); time.sleep(1)
    threading.Thread(target=_tail_loop, daemon=True).start()
    print('[tailer] streaming binary output + W&B metrics...', flush=True)

os.chdir(BUILD)
REPLAY_ARG = ['--replays', 'serialized_replays.bin'] if os.path.exists('serialized_replays.bin') else []
BASE_CMD = ['stdbuf', '-oL', '-eL', './R3maJ', '--device', DEVICE, '--phase', '-1', '--save-dir', 'checkpoints', '--games', '164'] + REPLAY_ARG
print('BASE CMD:', ' '.join(BASE_CMD), flush=True)

backoff = 10
attempt = 0
with open('train.log', 'a') as log:
    while True:
        attempt += 1
        t0 = time.time()
        print(f'[supervisor] launch #{attempt}', flush=True)
        proc = subprocess.Popen(BASE_CMD, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        rc = proc.wait()
        elapsed = time.time() - t0
        if rc == 0:
            print('[supervisor] clean exit (rc=0)', flush=True)
            break
        print(f'[supervisor] crashed rc={rc} after {elapsed:.0f}s — restarting in {backoff}s', flush=True)
        time.sleep(backoff)
        backoff = min(backoff * 2, 600) if elapsed < 60 else 10
print('supervisor exited.')